# Homography correctness desc stats

In [25]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import itertools
import scipy.stats as stats
# ============================================================
# Display options
# ============================================================
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

# ============================================================
# Evaluation directories
# ============================================================
# eval_log_dirs = [
#     "/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_v16.2-chroma-ISPDefaultInitialHype_original_HPatchesV4.1",
#     "/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_sunlit_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_adjust_defaultcolorhuesat_120000_HPatchesV4.1",
#     "/home/boat/proxyISP/pytorch-superpoint/logs/eval_sl_PRETRAINED_SAMEMODELHOMOADAPT_AGGRESSIVEHOMOADAPT_CFANORMALIZE_XHOMOWARP_DETERMHOMOADAPT_train_v16.2-chroma-HumanTunedInitialHype_sunlit_lr0.005_bothLoss_initialHypeHomoAdaptOnly_gradac187_78500_HPatchesV4.1",
# ]

eval_log_dirs = [
    "/home/boat/proxyISP/pytorch-superpoint/logs/eval_ll_v16.2-chroma-ISPDefaultInitialHype_original_HPatchesV4.1",
    "/home/boat/proxyISP/pytorch-superpoint/logs/eval_ll_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_lowlight_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_adjust_defaultcolorhuesat_denoise_45000_HPatchesV4.1",
    "/home/boat/proxyISP/pytorch-superpoint/logs/eval_ll_PRETRAINED_SAMEMODELHOMOADAPT_AGGRESSIVEHOMOADAPT_CFANORMALIZE_XHOMOWARP_DETERMHOMOADAPT_train_v16.2-chroma-HumanTunedInitialHype_lowlight_lr0.005_bothLoss_initialHypeHomoAdaptOnly_gradac187_90200_HPatchesV4.1",
]

suffix = "ll"

names = [
    "Original ISP",
    "Visual-Optimized ISP",
    "Feature-Optimized ISP",
]

# ============================================================
# Load result.npz files
# ============================================================
objects = {}

for k,d in enumerate(eval_log_dirs):

    name = names[k]

    result_path = Path(d) / "predictions" / "result.npz"

    if not result_path.exists():
        raise FileNotFoundError(result_path)

    objects[name] = np.load(result_path)

print("=" * 80)
print("Loaded evaluations")
print("=" * 80)

for k in objects:
    print(k)

# ============================================================
# Print descriptive statistics
# ============================================================
first_name = next(iter(objects))
thresholds = objects[first_name]["homography_thresh"]

print("\n" + "=" * 80)
print(f"Homography Correctness Descriptive Statistics suffix:{suffix}")
print("=" * 80)

all_tables = []

for k_idx, threshold in enumerate(thresholds):

    print(f"\nThreshold @ {threshold}")
    print("-" * 80)

    stats_df = pd.DataFrame()

    for eval_name, obj in objects.items():

        # ----------------------------------------------------
        # correctness shape:
        # [num_pairs, num_thresholds]
        # ----------------------------------------------------
        correctness = obj["correctness"][:, k_idx]

        # bool -> float
        correctness = correctness.astype(np.float32)

        # ----------------------------------------------------
        # descriptive statistics
        # ----------------------------------------------------
        mean_ = correctness.mean()
        std_ = correctness.std()
        median_ = np.median(correctness)
        min_ = correctness.min()
        max_ = correctness.max()

        success_count = int(correctness.sum())
        total_count = len(correctness)

        success_rate = mean_ * 100

        # ----------------------------------------------------
        # save table row
        # ----------------------------------------------------
        stats_df.at[eval_name, "mean"] = mean_
        stats_df.at[eval_name, "std"] = std_
        stats_df.at[eval_name, "median"] = median_
        stats_df.at[eval_name, "min"] = min_
        stats_df.at[eval_name, "max"] = max_
        stats_df.at[eval_name, "success_count"] = success_count
        stats_df.at[eval_name, "total_pairs"] = total_count
        stats_df.at[eval_name, "success_rate_%"] = success_rate

        # ----------------------------------------------------
        # pretty print
        # ----------------------------------------------------
        print(
            f"{eval_name}"
            f"\n    mean           : {mean_:.4f}"
            f"\n    std            : {std_:.4f}"
            f"\n    median         : {median_:.4f}"
            f"\n    min            : {min_:.4f}"
            f"\n    max            : {max_:.4f}"
            f"\n    success_count  : {success_count}"
            f"\n    total_pairs    : {total_count}"
            f"\n    success_rate   : {success_rate:.2f}%"
            "\n"
        )

    stats_df = stats_df.round(4)

    all_tables.append((threshold, stats_df))

    print(stats_df)

# ============================================================
# Optional: save all stats to CSV
# ============================================================
save_csv = False

if save_csv:

    save_dir = Path("homography_stats")
    save_dir.mkdir(exist_ok=True)

    for threshold, df in all_tables:

        csv_path = save_dir / f"homography_stats_threshold_{threshold}.csv"

        df.to_csv(csv_path)

        print(f"Saved: {csv_path}")

Loaded evaluations
Original ISP
Visual-Optimized ISP
Feature-Optimized ISP

Homography Correctness Descriptive Statistics suffix:ll

Threshold @ 1
--------------------------------------------------------------------------------
Original ISP
    mean           : 0.2593
    std            : 0.4382
    median         : 0.0000
    min            : 0.0000
    max            : 1.0000
    success_count  : 35
    total_pairs    : 135
    success_rate   : 25.93%

Visual-Optimized ISP
    mean           : 0.2222
    std            : 0.4157
    median         : 0.0000
    min            : 0.0000
    max            : 1.0000
    success_count  : 30
    total_pairs    : 135
    success_rate   : 22.22%

Feature-Optimized ISP
    mean           : 0.2519
    std            : 0.4341
    median         : 0.0000
    min            : 0.0000
    max            : 1.0000
    success_count  : 34
    total_pairs    : 135
    success_rate   : 25.19%

                         mean     std  median  min  max  succe

# Homography Correctness hyp test

In [26]:
# ============================================================
# Get thresholds
# ============================================================
first = next(iter(objects))
thresholds = objects[first]["homography_thresh"]

# ============================================================
# McNemar test helper
# ============================================================
def mcnemar_test(x, y):
    """
    x, y: binary arrays (0/1)
    """

    x = x.astype(bool)
    y = y.astype(bool)

    # contingency table
    # b: x correct, y wrong
    # c: x wrong, y correct
    b = np.sum((x == 1) & (y == 0))
    c = np.sum((x == 0) & (y == 1))

    # avoid division by zero
    if b + c == 0:
        return 0.0, 1.0

    # McNemar chi-square (with continuity correction)
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    p = 1 - stats.chi2.cdf(chi2, df=1)

    return chi2, p


# ============================================================
# Paired proportion z-test
# ============================================================
def paired_z_test(x, y):
    x = x.astype(float)
    y = y.astype(float)

    d = x - y

    mean_d = np.mean(d)
    std_d = np.std(d, ddof=1)
    n = len(d)

    if std_d == 0:
        return 0.0, 1.0

    z = mean_d / (std_d / np.sqrt(n))
    p = 2 * (1 - stats.norm.cdf(abs(z)))

    return z, p


# ============================================================
# Hypothesis testing
# ============================================================
print("\n" + "=" * 80)
print(f"Homography Correctness Hypothesis Testing: {suffix}")
print("=" * 80)

keys = list(objects.keys())

for k_idx, thr in enumerate(thresholds):

    print(f"\nThreshold @ {thr}")
    print("-" * 80)

    for a, b in itertools.combinations(keys, 2):

        A = objects[a]["correctness"][:, k_idx]
        B = objects[b]["correctness"][:, k_idx]

        A = A.astype(np.int32)
        B = B.astype(np.int32)

        # ----------------------------------------------------
        # McNemar test (best for binary paired data)
        # ----------------------------------------------------
        chi2, p_mcn = mcnemar_test(A, B)

        # ----------------------------------------------------
        # Paired z-test
        # ----------------------------------------------------
        z, p_z = paired_z_test(A, B)

        # ----------------------------------------------------
        # simple effect size
        # ----------------------------------------------------
        diff = A.mean() - B.mean()

        print(f"\n{a} vs {b}")
        print(f"  mean(A) - mean(B): {diff:.5f}")
        print(f"  McNemar chi2     : {chi2:.4f}")
        print(f"  McNemar p-value  : {p_mcn:.6f}")
        print(f"  z-statistic      : {z:.4f}")
        print(f"  z-test p-value   : {p_z:.6f}")

        if p_mcn < 0.05:
            print("  McNemar result   : SIGNIFICANT")
        else:
            print("  McNemar result   : NOT significant")


Homography Correctness Hypothesis Testing: ll

Threshold @ 1
--------------------------------------------------------------------------------

Original ISP vs Visual-Optimized ISP
  mean(A) - mean(B): 0.03704
  McNemar chi2     : 1.2308
  McNemar p-value  : 0.267257
  z-statistic      : 1.3916
  z-test p-value   : 0.164058
  McNemar result   : NOT significant

Original ISP vs Feature-Optimized ISP
  mean(A) - mean(B): 0.00741
  McNemar chi2     : 0.0000
  McNemar p-value  : 1.000000
  z-statistic      : 0.1993
  z-test p-value   : 0.842038
  McNemar result   : NOT significant

Visual-Optimized ISP vs Feature-Optimized ISP
  mean(A) - mean(B): -0.02963
  McNemar chi2     : 0.4500
  McNemar p-value  : 0.502335
  z-statistic      : -0.8938
  z-test p-value   : 0.371450
  McNemar result   : NOT significant

Threshold @ 3
--------------------------------------------------------------------------------

Original ISP vs Visual-Optimized ISP
  mean(A) - mean(B): 0.02222
  McNemar chi2     : 0

# Loc Err desc stats

In [27]:
# ============================================================
# Generic descriptive statistics utility
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Helper: stats function
# ============================================================
def compute_stats(x):
    x = np.asarray(x).reshape(-1)

    return {
        "mean": np.mean(x),
        "std": np.std(x),
        "median": np.median(x),
        "min": np.min(x),
        "max": np.max(x),
        "rmse": np.sqrt(np.mean(x**2)),
        "p95": np.percentile(x, 95),
        "p90": np.percentile(x, 90),
    }


# ============================================================
# Main function
# ============================================================
def summarize_metric(objects, key, suffix=""):
    """
    Compute descriptive statistics for a given key inside objects.

    Parameters
    ----------
    objects : dict
        Dictionary of experiment/object results.

    key : str
        Metric key to analyze.
        Example:
            "localization_err"
            "rotation_err"
            "confidence"

    suffix : str
        Optional suffix for printing.
    """

    print("\n" + "=" * 80)
    print(f"{key} Descriptive Statistics {suffix}")
    print("=" * 80)

    all_results = {}

    for name, obj in objects.items():

        if key not in obj:
            raise KeyError(
                f"{key} not found in {name}. "
                f"Available keys: {list(obj.keys())}"
            )

        values = obj[key]

        stats = compute_stats(values)

        all_results[name] = stats

        print(f"\n{name}")
        print("-" * 80)

        for k, v in stats.items():
            print(f"{k:10s}: {v:.6f}")

    # ========================================================
    # Summary table
    # ========================================================
    df = pd.DataFrame(all_results).T
    df = df.round(6)

    print("\n" + "=" * 80)
    print("Summary Table")
    print("=" * 80)

    print(df)

    return df


# ============================================================
# Example usage
# ============================================================

# localization error
df_loc = summarize_metric(
    objects,
    key="localization_err",
    suffix="(Localization)"
)


localization_err Descriptive Statistics (Localization)

Original ISP
--------------------------------------------------------------------------------
mean      : 1.085099
std       : 0.268972
median    : 1.072918
min       : 0.587690
max       : 1.845188
rmse      : 1.117938
p95       : 1.512985
p90       : 1.436101

Visual-Optimized ISP
--------------------------------------------------------------------------------
mean      : 1.082478
std       : 0.264737
median    : 1.054982
min       : 0.607286
max       : 1.795599
rmse      : 1.114380
p95       : 1.539464
p90       : 1.418980

Feature-Optimized ISP
--------------------------------------------------------------------------------
mean      : 1.072326
std       : 0.266992
median    : 1.046499
min       : 0.574509
max       : 1.770095
rmse      : 1.105064
p95       : 1.491043
p90       : 1.431593

Summary Table
                           mean       std    median       min       max      rmse       p95       p90
Original ISP         

# Loc Err hyp test

In [28]:
import itertools
import numpy as np
import pandas as pd
import scipy.stats as stats


# ============================================================
# Statistical test helpers
# ============================================================
def wilcoxon_test(x, y):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    try:
        stat, p = stats.wilcoxon(x, y, alternative="two-sided")
    except ValueError:
        # occurs if all paired differences are zero
        return 0.0, 1.0

    return stat, p


def paired_t_test(x, y):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    stat, p = stats.ttest_rel(x, y)

    return stat, p


# ============================================================
# Main hypothesis testing function
# ============================================================
def hypothesis_testing_ttest_wilcox(objects, key, suffix="", alpha=0.05):
    """
    Perform pairwise statistical hypothesis testing.

    Parameters
    ----------
    objects : dict
        Dictionary containing experiment results.

    key : str
        Metric key to compare.
        Example:
            "localization_err"
            "rotation_err"

    suffix : str
        Optional title suffix.

    alpha : float
        Significance threshold.
    """

    print("\n" + "=" * 80)
    print(f"{key} Hypothesis Testing {suffix}")
    print("=" * 80)

    keys = list(objects.keys())

    summary_rows = []

    for a, b in itertools.combinations(keys, 2):

        if key not in objects[a]:
            raise KeyError(f"{key} not found in {a}")

        if key not in objects[b]:
            raise KeyError(f"{key} not found in {b}")

        A = np.asarray(objects[a][key]).reshape(-1)
        B = np.asarray(objects[b][key]).reshape(-1)

        # ----------------------------------------------------
        # paired assumption
        # ----------------------------------------------------
        if len(A) != len(B):
            raise ValueError(
                f"Mismatch length: {a}={len(A)}, {b}={len(B)}"
            )

        # ----------------------------------------------------
        # effect size
        # ----------------------------------------------------
        mean_diff = A.mean() - B.mean()

        # ----------------------------------------------------
        # Wilcoxon
        # ----------------------------------------------------
        w_stat, w_p = wilcoxon_test(A, B)

        # ----------------------------------------------------
        # Paired t-test
        # ----------------------------------------------------
        t_stat, t_p = paired_t_test(A, B)

        # ----------------------------------------------------
        # significance
        # ----------------------------------------------------
        significant = w_p < alpha

        # ----------------------------------------------------
        # print
        # ----------------------------------------------------
        print(f"\n{a} vs {b}")
        print("-" * 60)

        print(f"Mean(A) - Mean(B): {mean_diff:.6f}")

        print("\nWilcoxon signed-rank test")
        print(f"  statistic : {w_stat:.6f}")
        print(f"  p-value   : {w_p:.6e}")

        print("\nPaired t-test")
        print(f"  t-stat    : {t_stat:.6f}")
        print(f"  p-value   : {t_p:.6e}")

        if significant:
            print("\nResult: SIGNIFICANT difference (Wilcoxon)")
        else:
            print("\nResult: NOT significant (Wilcoxon)")

        # ----------------------------------------------------
        # summary row
        # ----------------------------------------------------
        summary_rows.append({
            "A": a,
            "B": b,
            "mean_diff": mean_diff,
            "wilcoxon_stat": w_stat,
            "wilcoxon_p": w_p,
            "t_stat": t_stat,
            "t_p": t_p,
            "significant": significant,
        })

    # ========================================================
    # summary dataframe
    # ========================================================
    df = pd.DataFrame(summary_rows)

    print("\n" + "=" * 80)
    print("Summary Table")
    print("=" * 80)

    print(df)

    return df


# ============================================================
# Example usage
# ============================================================

df_loc = hypothesis_testing_ttest_wilcox(
    objects,
    key="localization_err",
    suffix="(Localization)"
)


localization_err Hypothesis Testing (Localization)

Original ISP vs Visual-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): 0.002621

Wilcoxon signed-rank test
  statistic : 4423.000000
  p-value   : 7.137860e-01

Paired t-test
  t-stat    : 0.559351
  p-value   : 5.768561e-01

Result: NOT significant (Wilcoxon)

Original ISP vs Feature-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): 0.012773

Wilcoxon signed-rank test
  statistic : 3293.000000
  p-value   : 4.391826e-03

Paired t-test
  t-stat    : 3.239054
  p-value   : 1.512249e-03

Result: SIGNIFICANT difference (Wilcoxon)

Visual-Optimized ISP vs Feature-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): 0.010152

Wilcoxon signed-rank test
  statistic : 3140.000000
  p-value   : 1.449625e-03

Paired t-test
  t-stat    : 2.871113
  p-value   : 4.755806e-03

Result: SIGNIFICANT difference (Wilcoxon)


# Repeatability

In [29]:
df_loc = summarize_metric(
    objects,
    key="repeatability",
    suffix="(Localization)"
)


repeatability Descriptive Statistics (Localization)

Original ISP
--------------------------------------------------------------------------------
mean      : 0.638731
std       : 0.137465
median    : 0.658171
min       : 0.253555
max       : 0.863535
rmse      : 0.653356
p95       : 0.823523
p90       : 0.797041

Visual-Optimized ISP
--------------------------------------------------------------------------------
mean      : 0.643788
std       : 0.137933
median    : 0.662791
min       : 0.229698
max       : 0.847548
rmse      : 0.658398
p95       : 0.828987
p90       : 0.799039

Feature-Optimized ISP
--------------------------------------------------------------------------------
mean      : 0.644269
std       : 0.140351
median    : 0.664122
min       : 0.223650
max       : 0.855972
rmse      : 0.659380
p95       : 0.822847
p90       : 0.807732

Summary Table
                           mean       std    median       min       max      rmse       p95       p90
Original ISP           0

# Repeatability hyp test

In [30]:
df_loc = hypothesis_testing_ttest_wilcox(
    objects,
    key="repeatability",
    suffix="(Localization)"
)


repeatability Hypothesis Testing (Localization)

Original ISP vs Visual-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): -0.005057

Wilcoxon signed-rank test
  statistic : 3715.000000
  p-value   : 5.463956e-02

Paired t-test
  t-stat    : -2.341303
  p-value   : 2.069140e-02

Result: NOT significant (Wilcoxon)

Original ISP vs Feature-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): -0.005538

Wilcoxon signed-rank test
  statistic : 3444.000000
  p-value   : 1.183860e-02

Paired t-test
  t-stat    : -2.886859
  p-value   : 4.537561e-03

Result: SIGNIFICANT difference (Wilcoxon)

Visual-Optimized ISP vs Feature-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): -0.000481

Wilcoxon signed-rank test
  statistic : 4261.000000
  p-value   : 4.699435e-01

Paired t-test
  t-stat    : -0.334136
  p-value   : 7.387991e-01

Result: NOT significant (Wilcoxon)

Sum

# mAP

In [31]:
df_loc = summarize_metric(
    objects,
    key="mAP",
    suffix="(Localization)"
)


mAP Descriptive Statistics (Localization)

Original ISP
--------------------------------------------------------------------------------
mean      : 0.747427
std       : 0.236712
median    : 0.813235
min       : 0.066075
max       : 0.995108
rmse      : 0.784015
p95       : 0.976526
p90       : 0.967313

Visual-Optimized ISP
--------------------------------------------------------------------------------
mean      : 0.734774
std       : 0.235975
median    : 0.795068
min       : 0.065044
max       : 0.992165
rmse      : 0.771736
p95       : 0.974723
p90       : 0.957020

Feature-Optimized ISP
--------------------------------------------------------------------------------
mean      : 0.744920
std       : 0.230746
median    : 0.796819
min       : 0.056432
max       : 0.990861
rmse      : 0.779839
p95       : 0.981955
p90       : 0.969017

Summary Table
                           mean       std    median       min       max      rmse       p95       p90
Original ISP           0.747427  0

In [32]:
df_loc = hypothesis_testing_ttest_wilcox(
    objects,
    key="mAP",
    suffix="(Localization)"
)


mAP Hypothesis Testing (Localization)

Original ISP vs Visual-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): 0.012653

Wilcoxon signed-rank test
  statistic : 3764.000000
  p-value   : 6.966011e-02

Paired t-test
  t-stat    : 1.636935
  p-value   : 1.039904e-01

Result: NOT significant (Wilcoxon)

Original ISP vs Feature-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): 0.002507

Wilcoxon signed-rank test
  statistic : 4058.000000
  p-value   : 2.426403e-01

Paired t-test
  t-stat    : 0.291569
  p-value   : 7.710667e-01

Result: NOT significant (Wilcoxon)

Visual-Optimized ISP vs Feature-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): -0.010146

Wilcoxon signed-rank test
  statistic : 4202.000000
  p-value   : 3.941305e-01

Paired t-test
  t-stat    : -1.378791
  p-value   : 1.702564e-01

Result: NOT significant (Wilcoxon)

Summary Table
          

# Match Score

In [33]:
df_loc = summarize_metric(
    objects,
    key="mscore",
    suffix="(Localization)"
)


mscore Descriptive Statistics (Localization)

Original ISP
--------------------------------------------------------------------------------
mean      : 0.338194
std       : 0.242801
median    : 0.296925
min       : 0.019152
max       : 0.827667
rmse      : 0.416326
p95       : 0.768096
p90       : 0.730184

Visual-Optimized ISP
--------------------------------------------------------------------------------
mean      : 0.338378
std       : 0.241519
median    : 0.279891
min       : 0.017266
max       : 0.855280
rmse      : 0.415729
p95       : 0.769737
p90       : 0.742564

Feature-Optimized ISP
--------------------------------------------------------------------------------
mean      : 0.341478
std       : 0.239712
median    : 0.295172
min       : 0.018127
max       : 0.862385
rmse      : 0.417216
p95       : 0.750814
p90       : 0.708728

Summary Table
                           mean       std    median       min       max      rmse       p95       p90
Original ISP           0.338194

In [34]:
df_loc = hypothesis_testing_ttest_wilcox(
    objects,
    key="mscore",
    suffix="(Localization)"
)


mscore Hypothesis Testing (Localization)

Original ISP vs Visual-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): -0.000184

Wilcoxon signed-rank test
  statistic : 4416.000000
  p-value   : 7.023501e-01

Paired t-test
  t-stat    : -0.058728
  p-value   : 9.532564e-01

Result: NOT significant (Wilcoxon)

Original ISP vs Feature-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): -0.003285

Wilcoxon signed-rank test
  statistic : 4120.000000
  p-value   : 3.019577e-01

Paired t-test
  t-stat    : -1.092934
  p-value   : 2.763839e-01

Result: NOT significant (Wilcoxon)

Visual-Optimized ISP vs Feature-Optimized ISP
------------------------------------------------------------
Mean(A) - Mean(B): -0.003101

Wilcoxon signed-rank test
  statistic : 4018.000000
  p-value   : 2.090205e-01

Paired t-test
  t-stat    : -0.983866
  p-value   : 3.269547e-01

Result: NOT significant (Wilcoxon)

Summary Table
   